# PyTorX - MNIST Benchmark

This notebook trains a LeNet-style network on MNIST using PyTorX crossbar-aware layers.

**Non-ideal effects available:** IR drop, thermal/RTN noise, stuck-at-fault (SAF), DAC/ADC quantization.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fouda-compumacy/pytorx/blob/alpha/notebooks/mnist.ipynb)

## 1. Setup

In [ ]:
import os

# Clone PyTorX if not already present
if not os.path.exists('pytorx'):
    !git clone https://github.com/fouda-compumacy/pytorx.git

os.chdir('pytorx')

# Install dependencies
!pip install -q torch torchvision numpy scipy pandas matplotlib

In [ ]:
import sys
import shutil
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from IPython.display import clear_output

from python.torx.module.layer import crxb_Conv2d, crxb_Linear

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Configuration

Adjust crossbar parameters and non-ideal effect toggles here.

In [ ]:
class Config:
    # Training
    epochs = 20
    batch_size = 1000
    test_batch_size = 100
    lr = 0.5
    momentum = 0.5
    seed = 1

    # Crossbar
    crxb_size = 64
    vdd = 3.3
    gwire = 0.375
    gload = 0.25
    gmax = 0.000333
    gmin = 0.0000000333
    freq = 10e6
    temp = 300
    scaler_dw = 1
    quantize = 8
    weight_precision = None  # None = same as quantize

    # Non-ideal effects (toggle these for experiments)
    ir_drop = False
    enable_noise = False
    enable_SAF = False
    enable_ec_SAF = False

args = Config()
torch.manual_seed(args.seed)

## 3. Data Loading

In [ ]:
kwargs = {'num_workers': 2, 'pin_memory': True} if torch.cuda.is_available() else {}

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=True, download=True,
                   transform=transforms.Compose([
                       transforms.ToTensor(),
                       transforms.Normalize((0.1307,), (0.3081,))
                   ])),
    batch_size=args.batch_size, shuffle=True, **kwargs)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=False,
                   transform=transforms.Compose([
                       transforms.ToTensor(),
                       transforms.Normalize((0.1307,), (0.3081,))
                   ])),
    batch_size=args.test_batch_size, shuffle=False, **kwargs)

print(f'Train: {len(train_loader.dataset)} samples, Test: {len(test_loader.dataset)} samples')

## 4. Model Definition

In [ ]:
class Net(nn.Module):
    def __init__(self, cfg):
        super(Net, self).__init__()
        crxb_cfg = dict(
            crxb_size=cfg.crxb_size, scaler_dw=cfg.scaler_dw,
            gwire=cfg.gwire, gload=cfg.gload, gmax=cfg.gmax, gmin=cfg.gmin,
            vdd=cfg.vdd, freq=cfg.freq, temp=cfg.temp,
            enable_SAF=cfg.enable_SAF, enable_ec_SAF=cfg.enable_ec_SAF,
            enable_noise=cfg.enable_noise, ir_drop=cfg.ir_drop,
            device=device, quantize=cfg.quantize,
            weight_precision=cfg.weight_precision)

        self.conv1 = crxb_Conv2d(1, 10, kernel_size=5, **crxb_cfg)
        self.conv2 = crxb_Conv2d(10, 20, kernel_size=5, **crxb_cfg)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = crxb_Linear(320, 50, **crxb_cfg)
        self.fc2 = crxb_Linear(50, 10, **crxb_cfg)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


model = Net(args).to(device)
print(model)

## 5. Training

In [ ]:
optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=args.momentum)
criterion = nn.NLLLoss().to(device)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.2, patience=2,
    threshold=0.5, threshold_mode='rel', min_lr=1e-4)


def train_epoch(model, loader, optimizer, criterion, epoch):
    model.train()
    total_loss, correct = 0.0, 0
    for batch_idx, (data, target) in enumerate(loader):
        for m in model.modules():
            if isinstance(m, (crxb_Conv2d, crxb_Linear)):
                m._reset_delta()
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.size(0)
        correct += output.argmax(1).eq(target).sum().item()
    n = len(loader.dataset)
    return total_loss / n, 100. * correct / n


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_loss += criterion(output, target).item() * data.size(0)
            correct += output.argmax(1).eq(target).sum().item()
    n = len(loader.dataset)
    return total_loss / n, 100. * correct / n

In [ ]:
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(1, args.epochs + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, epoch)
    val_loss, val_acc = evaluate(model, test_loader, criterion)
    scheduler.step(val_loss)

    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    train_accs.append(tr_acc)
    val_accs.append(val_acc)

    # Live plot
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(train_losses, label='Train')
    ax1.plot(val_losses, label='Val')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
    ax2.plot(train_accs, label='Train')
    ax2.plot(val_accs, label='Val')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.legend(); ax2.set_title('Accuracy')
    plt.tight_layout()
    plt.show()

    lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch}/{args.epochs}  LR={lr:.6f}  '
          f'Train Loss={tr_loss:.4f}  Train Acc={tr_acc:.2f}%  '
          f'Val Loss={val_loss:.4f}  Val Acc={val_acc:.2f}%')

    # Early stopping
    if lr < ((scheduler.min_lrs[0] / scheduler.factor) + scheduler.min_lrs[0]) / 2:
        print('Accuracy not improving, stopping.')
        break

## 6. Save Checkpoint

In [ ]:
torch.save({
    'epoch': epoch,
    'state_dict': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'val_loss': val_losses[-1],
    'val_acc': val_accs[-1],
}, f'checkpoint_mnist_{args.crxb_size}.pth.tar')
print(f'Saved checkpoint. Final val accuracy: {val_accs[-1]:.2f}%')